# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [10]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('sample_data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [11]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [12]:
import plotly.graph_objects as go

# Filter for Asian countries
asia_df = df[df['Region'] == 'Asia']
asian_countries = ['China', 'India', 'Japan', 'South Korea', 'Indonesia']
highlight_country = 'China'

fig = go.Figure()

for country in asian_countries:
    country_data = asia_df[asia_df['Country'] == country].sort_values('Year')

    if country == highlight_country:
        # Highlighted country in bold red/orange with extra line width
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#D95F02', width=4),
            hoverinfo='text',
            text=[f"{country}: {y:.1f} Mt in {x}" for x, y in zip(country_data['Year'], country_data['CO2_Mt'])]
        ))
        # Direct label at the end of the highlighted line
        last_row = country_data.iloc[-1]
        fig.add_trace(go.Scatter(
            x=[last_row['Year']],
            y=[last_row['CO2_Mt']],
            mode='markers+text',
            text=[f"<b>{country}</b>"],
            textposition="middle right",
            marker=dict(color='#D95F02', size=8),
            showlegend=False,
            textfont=dict(size=12, color='#D95F02')
        ))
    else:
        # Other background countries in thin light grey
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#EEEEEE', width=1.5),
            hoverinfo='text',
            text=[f"{country}: {y:.1f} Mt in {x}" for x, y in zip(country_data['Year'], country_data['CO2_Mt'])]
        ))
        # Subtle label at the end of background lines for context
        last_row = country_data.iloc[-1]
        fig.add_trace(go.Scatter(
            x=[last_row['Year']],
            y=[last_row['CO2_Mt']],
            mode='text',
            text=[f"  {country}"],
            textposition="middle right",
            showlegend=False,
            textfont=dict(size=10, color='#999999')
        ))

# Configure a clean and professional layout
fig.update_layout(
    title={
        'text': "<b>China's CO2 Emissions Surge Far Past Regional Peers Since 2000</b><br><span style='font-size:13px; color:#555;'>Annual CO2 emissions in Megatonnes (Mt) from 2000 to 2022</span>",
        'y': 0.95,
        'x': 0.05,
        'xanchor': 'left',
        'yanchor': 'top'
    },
    xaxis=dict(
        showgrid=False,
        range=[2000, 2026],  # Padding for end-of-line labels
        dtick=5,
        linecolor='#CCCCCC'
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#F5F5F5',
        linecolor='#CCCCCC',
        title="CO2 Emissions (Mt)"
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    margin=dict(l=60, r=120, t=80, b=50),
    width=850,
    height=500
)

fig.show()